# EfficientNet-B0 transfer learning - EuroSAT

EN3150 Assignment 03: an ImageNet-pretrained lightweight model for comparison with custom Model B and MobileNetV2.

This notebook currently establishes the shared dataset contract and builds an ImageNet-pretrained EfficientNet-B0 classifier. No training or test-performance evaluation is performed at this stage.

## Shared data and preprocessing

Use the exact committed `data/eurosat_split.csv` and local folders created by `Dataset.ipynb`: 18,900 training, 4,050 validation and 4,050 test images, in 10 classes. Run this notebook from the repository root after dataset generation. Never re-split, resize or augment these images.

**EfficientNet-specific input contract:** native 64x64 RGB JPEGs become float32 tensors in **0-255**, rather than Model B's 0-1 tensors. Keras EfficientNet-B0 includes its own rescaling and normalization; do not divide by 255 or apply additional ImageNet normalization here.

Reference: https://keras.io/api/applications/efficientnet/efficientnet_models/


In [42]:
import os
# Keep downloaded Keras weights in the workspace, outside version control.
os.environ["KERAS_HOME"] = os.path.abspath("data/keras_cache")
# Set before numerical-library imports. Restart the kernel if they are already loaded.
THREAD_ENV = ["OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS"]
for name in THREAD_ENV:
    os.environ[name] = "1"

import hashlib
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = Path("data/eurosat")
SPLIT_CSV = Path("data/eurosat_split.csv")
IMG_SIZE = (64, 64)
SEED = 42
BATCH_SIZE = 4  # Conservative starting batch; verify memory before full training.
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_COUNTS = {"train": 18_900, "val": 4_050, "test": 4_050}

tf.keras.utils.set_random_seed(SEED)
print("Python:", platform.python_version(), "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none")
print("Thread settings:", {name: os.environ.get(name) for name in THREAD_ENV})
print("Class mapping:", dict(enumerate(CLASS_NAMES)))


Python: 3.12.2 | TensorFlow: 2.21.0
GPU: none
Thread settings: {'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'TF_NUM_INTRAOP_THREADS': '1', 'TF_NUM_INTEROP_THREADS': '1'}
Class mapping: {0: 'AnnualCrop', 1: 'Forest', 2: 'HerbaceousVegetation', 3: 'Highway', 4: 'Industrial', 5: 'Pasture', 6: 'PermanentCrop', 7: 'Residential', 8: 'River', 9: 'SeaLake'}


## 1. Validate the shared split and files

The checks below reject missing/duplicate manifest identities, unexpected class or split names, wrong split proportions and extra/missing local files. File identities are `(class, filename)` pairs. A SHA-256 identifies the exact manifest for later experiment records; it is not an image-content checksum.


In [43]:
if not SPLIT_CSV.is_file() or not DATA_DIR.is_dir():
    raise FileNotFoundError("Run Dataset.ipynb from the repository root to create the shared EuroSAT dataset first.")
manifest = pd.read_csv(SPLIT_CSV, dtype=str)
assert list(manifest.columns) == ["file", "label", "split"], "Unexpected manifest columns"
assert not manifest.isna().any().any(), "Missing manifest values"
assert not manifest.duplicated(["label", "file"]).any(), "An image appears in more than one manifest row"
assert set(manifest["label"]) == set(CLASS_NAMES), "Unexpected class labels"
assert set(manifest["split"]) == set(EXPECTED_COUNTS), "Unexpected split names"
assert manifest["file"].map(lambda name: Path(name).name == name and name.lower().endswith(".jpg")).all(), "Invalid filenames"
assert len(manifest) == 27_000, "Expected 27,000 images"

counts = pd.crosstab(manifest["label"], manifest["split"]).reindex(
    index=CLASS_NAMES, columns=list(EXPECTED_COUNTS),
)
assert not counts.isna().any().any(), "A class is absent from a split"
np.testing.assert_allclose(
    counts.to_numpy() / counts.sum(axis=1).to_numpy()[:, None],
    np.tile([0.70, 0.15, 0.15], (NUM_CLASSES, 1)), rtol=0, atol=1e-12,
)
for split, expected in EXPECTED_COUNTS.items():
    assert int(counts[split].sum()) == expected, f"Unexpected {split} count"
expected_paths = {f"{row.split}/{row.label}/{row.file}" for row in manifest.itertuples(index=False)}
actual_paths = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()}
missing, extra = sorted(expected_paths - actual_paths), sorted(actual_paths - expected_paths)
assert not missing and not extra, f"Local files differ: missing={missing[:5]}, extra={extra[:5]}"
SPLIT_SHA256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print(counts.to_string())
print("Split totals:", counts.sum().to_dict())
print("Manifest SHA-256:", SPLIT_SHA256)


split                 train  val  test
label                                 
AnnualCrop             2100  450   450
Forest                 2100  450   450
HerbaceousVegetation   2100  450   450
Highway                1750  375   375
Industrial             1750  375   375
Pasture                1400  300   300
PermanentCrop          1750  375   375
Residential            2100  450   450
River                  1750  375   375
SeaLake                2100  450   450
Split totals: {'train': 18900, 'val': 4050, 'test': 4050}
Manifest SHA-256: 0cf4cc1ca5601a80a4a9c7ce47fb3435fcf95a36eba01802e0d08baf719f0c3c


## 2. Decode without resizing or external normalisation

Decode with the original channel count and enforce 64x64x3, so malformed inputs fail instead of being silently resized or converted. Shuffle paths before decoding for training and preserve sorted validation/test order. Each factory call creates a fresh shuffle state for future reproducible experiments.

Use bounded parallelism and a one-batch prefetch to limit host memory. Do not cache the whole decoded dataset. Reproducible seeds do not guarantee identical floating-point results across hardware or TensorFlow versions.


In [44]:
def decode_efficientnet_image(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=0)
    image = tf.ensure_shape(image, IMG_SIZE + (3,))
    return tf.cast(image, tf.float32), label  # Deliberately keep raw 0-255 pixels.


split_paths, split_labels = {}, {}
for split in EXPECTED_COUNTS:
    rows = manifest.loc[manifest["split"] == split].sort_values(["label", "file"])
    split_paths[split] = [str(DATA_DIR / row.split / row.label / row.file) for row in rows.itertuples(index=False)]
    split_labels[split] = np.array([CLASS_NAMES.index(label) for label in rows["label"]], dtype=np.int32)


def make_efficientnet_dataset(split, batch_size=BATCH_SIZE, shuffle=None):
    """Create a new bounded-memory pipeline; shuffle only training by default."""
    if split not in EXPECTED_COUNTS:
        raise ValueError(f"Unknown split: {split}")
    if not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("batch_size must be a positive integer")
    if shuffle is None:
        shuffle = split == "train"
    if split != "train" and shuffle:
        raise ValueError("Validation and test order must remain fixed")
    ds = tf.data.Dataset.from_tensor_slices((split_paths[split], split_labels[split]))
    if shuffle:
        ds = ds.shuffle(EXPECTED_COUNTS[split], seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.deterministic = True
    options.threading.private_threadpool_size = 2
    ds = ds.with_options(options)
    return ds.map(decode_efficientnet_image, num_parallel_calls=2).batch(batch_size).prefetch(1)


## 3. Validate every decoded input

This is structural data validation, including test images, not model evaluation. Check actual decoded counts, shapes, finite pixels and valid labels. An explicit byte-for-byte comparison on a training image catches accidental 0-1 scaling, which a range check alone would miss.


In [45]:
for split, expected in EXPECTED_COUNTS.items():
    seen = 0
    # Disable training shuffle for this inspection, leaving training factories fresh.
    for images, labels in make_efficientnet_dataset(split, shuffle=False):
        tf.debugging.assert_type(images, tf.float32)
        tf.debugging.assert_equal(tf.shape(images)[1:], [64, 64, 3])
        tf.debugging.assert_all_finite(images, "Non-finite input pixels")
        tf.debugging.assert_greater_equal(images, 0.0)
        tf.debugging.assert_less_equal(images, 255.0)
        tf.debugging.assert_greater_equal(labels, 0)
        tf.debugging.assert_less(labels, NUM_CLASSES)
        seen += int(tf.shape(images)[0])
    assert seen == expected, f"Wrong decoded count for {split}"
    print(f"{split}: {seen:,} valid 64x64 RGB images")

sample_path = split_paths["train"][0]
original = tf.io.decode_jpeg(tf.io.read_file(sample_path), channels=0)
sample_batch, sample_labels = next(iter(make_efficientnet_dataset("train", shuffle=False)))
np.testing.assert_array_equal(sample_batch[0].numpy(), tf.cast(original, tf.float32).numpy())
assert float(tf.reduce_max(sample_batch[0])) > 1.0, "Raw pixel scale was lost"
assert int(sample_labels[0]) == int(split_labels["train"][0])
print("Training sample range:", float(tf.reduce_min(sample_batch[0])), "to", float(tf.reduce_max(sample_batch[0])))
print("Raw pixel values preserved exactly; no external normalization applied.")

train_ds = make_efficientnet_dataset("train")
val_ds = make_efficientnet_dataset("val")
test_ds = make_efficientnet_dataset("test")
print("Data pipeline ready. No model built or training started.")


train: 18,900 valid 64x64 RGB images
val: 4,050 valid 64x64 RGB images
test: 4,050 valid 64x64 RGB images
Training sample range: 81.0 to 206.0
Raw pixel values preserved exactly; no external normalization applied.
Data pipeline ready. No model built or training started.


## 4. ImageNet-pretrained backbone and EuroSAT head

Load EfficientNet-B0 with `include_top=False`, ImageNet weights and native 64x64 RGB inputs. Replace its ImageNet classifier with global average pooling, dropout 0.3 and a ten-class softmax layer. Retain the pretrained model's internal rescaling/normalization: inputs remain raw 0-255 float32.

Freeze the entire backbone for the initial head-training stage. Call it with `training=False` to keep BatchNorm moving statistics and internal stochastic layers in inference mode, including during subsequent fine-tuning. The head's dropout still follows the outer model's training mode. Later we will explicitly unfreeze selected convolutional blocks, keep BatchNorm frozen and recompile with a fresh low-learning-rate optimizer.

64x64 inputs satisfy this implementation's minimum size, but ImageNet weights were trained at a different resolution and on natural photographs. Transfer quality on satellite patches must be measured; it is not guaranteed by the pretrained model's ImageNet accuracy. The assignment's 100,000-trainable-parameter cap applies to custom Model B, not this pretrained baseline.

Weights are cached in ignored `data/keras_cache/models/`. The first run needs internet access. Use `weights="imagenet"` rather than silently falling back to random weights if downloading fails.


In [46]:
from tensorflow.keras import layers, models


def build_efficientnet_b0(num_classes=NUM_CLASSES, dropout_rate=0.3):
    """Build a frozen ImageNet backbone with a freshly initialized EuroSAT head."""
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,),
    )
    backbone.trainable = False
    inputs = layers.Input(shape=IMG_SIZE + (3,), name="rgb_pixels_0_255")
    features = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(features)
    x = layers.Dropout(dropout_rate, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="land_use_probabilities")(x)
    model = models.Model(inputs, outputs, name="EuroSAT_EfficientNetB0")
    return model, backbone


tf.keras.utils.set_random_seed(SEED)
efficientnet_model, efficientnet_backbone = build_efficientnet_b0()
efficientnet_model.summary()


Model: "EuroSAT_EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rgb_pixels_0_255 (InputLayer)   │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 2, 2, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ land_use_probabilities (Dense)  │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [47]:
n_trainable = int(sum(np.prod(weight.shape) for weight in efficientnet_model.trainable_weights))
n_non_trainable = int(sum(np.prod(weight.shape) for weight in efficientnet_model.non_trainable_weights))
n_total = int(efficientnet_model.count_params())
assert n_total == n_trainable + n_non_trainable
assert not efficientnet_backbone.trainable and not efficientnet_backbone.trainable_weights
assert efficientnet_model.input_shape == (None, 64, 64, 3)
assert efficientnet_model.output_shape == (None, NUM_CLASSES)
head = efficientnet_model.get_layer("land_use_probabilities")
expected_head_params = int(efficientnet_backbone.output_shape[-1]) * NUM_CLASSES + NUM_CLASSES
assert n_trainable == head.count_params() == expected_head_params
assert any(isinstance(layer, layers.Rescaling) for layer in efficientnet_backbone.layers)

# Interface validation only; no accuracy measurement or test prediction.
x_check, _ = next(iter(make_efficientnet_dataset("val", batch_size=4)))
probabilities = efficientnet_model(x_check, training=False).numpy()
assert probabilities.shape == (len(x_check), NUM_CLASSES)
assert np.isfinite(probabilities).all()
assert ((probabilities >= 0) & (probabilities <= 1)).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-5)

# The frozen backbone must not update BatchNorm statistics in outer training mode.
batchnorm = next(layer for layer in efficientnet_backbone.layers if isinstance(layer, layers.BatchNormalization))
mean_before, variance_before = batchnorm.moving_mean.numpy().copy(), batchnorm.moving_variance.numpy().copy()
_ = efficientnet_model(x_check[:1], training=True)
np.testing.assert_array_equal(batchnorm.moving_mean.numpy(), mean_before)
np.testing.assert_array_equal(batchnorm.moving_variance.numpy(), variance_before)
print(f"Total parameters: {n_total:,}")
print(f"Trainable parameters (head only): {n_trainable:,}")
print(f"Non-trainable parameters: {n_non_trainable:,}")
print("Backbone feature-map shape:", efficientnet_backbone.output_shape)
print("Classifier output shape:", probabilities.shape)
print("Output probabilities and frozen BatchNorm checks passed. No training performed.")


Total parameters: 4,062,381
Trainable parameters (head only): 12,810
Non-trainable parameters: 4,049,571
Backbone feature-map shape: (None, 2, 2, 1280)
Classifier output shape: (4, 10)
Output probabilities and frozen BatchNorm checks passed. No training performed.


In [48]:
# Official Keras no-top B0 weights; Keras also checks this file's published hash on download.
WEIGHTS_FILE = Path(os.environ["KERAS_HOME"]) / "models" / "efficientnetb0_notop.h5"
EXPECTED_WEIGHTS_MD5 = "50bc09e76180e00e4465e1a485ddc09d"
assert WEIGHTS_FILE.is_file(), "Expected pretrained weight cache was not found"
weight_bytes = WEIGHTS_FILE.read_bytes()
assert hashlib.md5(weight_bytes).hexdigest() == EXPECTED_WEIGHTS_MD5, "Pretrained weight checksum differs"
PRETRAINED_PROVENANCE = {
    "architecture": "EfficientNetB0", "pretrained_dataset": "ImageNet", "include_top": False,
    "input_shape": [64, 64, 3], "input_range": [0, 255], "head_dropout": 0.3,
    "weights_url": "https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5",
    "weights_md5": EXPECTED_WEIGHTS_MD5, "weights_sha256": hashlib.sha256(weight_bytes).hexdigest(),
    "tensorflow": tf.__version__, "keras": tf.keras.__version__,
    "python": platform.python_version(), "split_sha256": SPLIT_SHA256,
}
del weight_bytes
print(PRETRAINED_PROVENANCE)


{'architecture': 'EfficientNetB0', 'pretrained_dataset': 'ImageNet', 'include_top': False, 'input_shape': [64, 64, 3], 'input_range': [0, 255], 'head_dropout': 0.3, 'weights_url': 'https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5', 'weights_md5': '50bc09e76180e00e4465e1a485ddc09d', 'weights_sha256': '0683edcf6151811fb3ddd6f098ac48df644d74592c4919a5acdee1c37554add2', 'tensorflow': '2.21.0', 'keras': '3.15.1', 'python': '3.12.2', 'split_sha256': '0cf4cc1ca5601a80a4a9c7ce47fb3435fcf95a36eba01802e0d08baf719f0c3c'}


## 5. Staged transfer-learning workflow

1. **Head stage:** a fresh ImageNet backbone stays frozen while the new classifier learns.
2. **Fine-tuning stage:** reload the best head-stage checkpoint, unfreeze layers from a named upper block onward, keep every BatchNorm layer frozen, and compile with a **new Adam optimizer** at a lower learning rate. The backbone's `training=False` call is retained, so its moving statistics and stochastic layers stay in inference mode while eligible convolution weights can receive gradients.

All fine-tuning candidates must start from the same head checkpoint, not from the previous candidate. Only training and validation data enter these functions.

Each stage saves best and latest full-model checkpoints plus an epoch CSV in ignored `results/efficientnet_checkpoints/`. A completed-run JSON stores configuration, provenance, histories, timing and checkpoint hashes. Matching completed runs can be reused. Interrupted runs retain recovery files but are **not automatically resumed**: optimizer/scheduler/callback continuation needs explicit handling. Choose a new run ID or explicitly replace an old run to start over.

Comparison runs default to fixed budgets and constant learning rates. Scheduling and early stopping can be enabled later for a selected final recipe. A smoke run uses one epoch per stage on ten training and ten validation images; these metrics are not optimizer-selection evidence.


In [49]:
import json
import re
import time

STAGE_SCHEMA = 1
EXPERIMENT_DIR = Path("results/efficientnet_experiments")
CHECKPOINT_DIR = Path("results/efficientnet_checkpoints")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_stage_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def configure_upper_blocks(backbone, unfreeze_from):
    """Unfreeze the requested upper block and subsequent layers, except BatchNorm."""
    if not re.fullmatch(r"block[1-7][a-z]", unfreeze_from):
        raise ValueError("Use a block prefix such as 'block6a' or 'block7a'")
    first = next((i for i, layer in enumerate(backbone.layers) if layer.name.startswith(unfreeze_from + "_")), None)
    if first is None:
        raise ValueError(f"No EfficientNet block matches {unfreeze_from}")
    backbone.trainable = True
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= first and not isinstance(layer, layers.BatchNormalization)
    assert all(not layer.trainable for layer in backbone.layers if isinstance(layer, layers.BatchNormalization))
    assert backbone.trainable_weights, "No backbone weights were unfrozen"


class StageEpochRecorder(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.started = time.perf_counter()
        self.learning_rate = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))

    def on_epoch_end(self, epoch, logs=None):
        elapsed = float(time.perf_counter() - self.started)
        self.times.append(elapsed)
        if logs is not None:
            logs["seconds"] = elapsed
            logs["learning_rate"] = self.learning_rate


def stage_dataset(split, batch_size, smoke):
    if split not in ["train", "val"]:
        raise ValueError("Training stages only accept train or val")
    if not smoke:
        return make_efficientnet_dataset(split, batch_size=batch_size)
    # One image per class, selected without examining pixels or model predictions.
    indices = [int(np.flatnonzero(split_labels[split] == label)[0]) for label in range(NUM_CLASSES)]
    paths = [split_paths[split][i] for i in indices]
    labels = split_labels[split][indices]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if split == "train":
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(decode_efficientnet_image, num_parallel_calls=1).batch(batch_size).prefetch(1)


In [50]:
def run_transfer_stage(run_id, stage, learning_rate, epochs, batch_size=BATCH_SIZE,
                       head_run=None, unfreeze_from=None, smoke=False, schedule=False,
                       early_stopping=False, rerun=False, output_dir=EXPERIMENT_DIR,
                       checkpoint_dir=CHECKPOINT_DIR, verbose=2):
    """Train/reuse one stage; return its provenance record and best checkpoint path."""
    if not re.fullmatch(r"[a-z0-9][a-z0-9_-]*", run_id):
        raise ValueError("Use lowercase letters, digits, hyphens and underscores in run_id")
    if stage not in ["head", "finetune"] or not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("Expected head/finetune stage and positive finite learning rate")
    if not isinstance(epochs, int) or epochs < 1 or not isinstance(batch_size, int) or batch_size < 1:
        raise ValueError("epochs and batch_size must be positive integers")
    if smoke and (epochs != 1 or batch_size > 4):
        raise ValueError("Smoke stages require one epoch and batch size <=4")
    if stage == "head" and (head_run is not None or unfreeze_from is not None):
        raise ValueError("Head training starts fresh; no parent or unfreezing prefix is allowed")
    parent_hash = None
    parent_record = None
    if stage == "finetune":
        if head_run is None or unfreeze_from is None:
            raise ValueError("Fine-tuning requires a completed head run and upper-block prefix")
        parent_record = json.loads(Path(head_run).read_text(encoding="utf-8"))
        if parent_record.get("status") != "completed" or parent_record["metadata"]["stage"] != "head":
            raise ValueError("Parent must be a completed head stage")
        parent_meta = parent_record["metadata"]
        assert parent_meta["smoke"] == smoke, "Do not mix smoke and full stages"
        assert parent_meta["pretrained"] == PRETRAINED_PROVENANCE, "Parent provenance differs"
        assert parent_meta["class_names"] == CLASS_NAMES
        parent_hash = file_sha256(parent_record["best_checkpoint"])
        assert parent_hash == parent_record["best_checkpoint_sha256"], "Parent checkpoint changed"

    metadata = {
        "schema": STAGE_SCHEMA, "pipeline": "eurosat-raw255-bounded-v1",
        "run_id": run_id, "stage": stage, "learning_rate": float(learning_rate),
        "epochs": epochs, "batch_size": batch_size, "seed": SEED, "smoke": smoke,
        "unfreeze_from": unfreeze_from, "schedule": schedule, "early_stopping": early_stopping,
        "parent_checkpoint_sha256": parent_hash, "pretrained": PRETRAINED_PROVENANCE,
        "class_names": CLASS_NAMES, "optimizer": "Adam",
        "sample_counts": {"train": 10, "val": 10} if smoke else {k: EXPECTED_COUNTS[k] for k in ["train", "val"]},
        "platform": platform.platform(), "processor": platform.processor(),
        "devices": [device.name for device in tf.config.list_physical_devices()],
        "thread_environment": {name: os.environ.get(name) for name in THREAD_ENV},
    }
    output_dir = Path(output_dir)
    record_path = output_dir / f"{run_id}.json"
    run_dir = Path(checkpoint_dir) / run_id
    best_path, latest_path = run_dir / "best_full.keras", run_dir / "latest_full.keras"
    if record_path.exists() and not rerun:
        cached = json.loads(record_path.read_text(encoding="utf-8"))
        if cached.get("status") != "completed" or cached.get("metadata") != metadata:
            raise ValueError("Cached stage differs; use a new run ID or rerun=True")
        assert file_sha256(cached["best_checkpoint"]) == cached["best_checkpoint_sha256"]
        print("Reusing completed stage:", record_path)
        return cached
    if run_dir.exists() and any(run_dir.iterdir()) and not rerun:
        raise RuntimeError("Checkpoint files exist without a reusable record; inspect them before rerunning")
    run_dir.mkdir(parents=True, exist_ok=True)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    if stage == "head":
        model, backbone = build_efficientnet_b0()
    else:
        model = tf.keras.models.load_model(parent_record["best_checkpoint"], compile=False)
        backbone = model.get_layer("efficientnetb0")
        configure_upper_blocks(backbone, unfreeze_from)
    # Recompile with a fresh optimizer after changing trainability; no inherited momentum slots.
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    trainable = int(sum(np.prod(weight.shape) for weight in model.trainable_weights))
    if stage == "head":
        assert trainable == 12_810 and not backbone.trainable_weights
    else:
        assert trainable > 12_810
    recorder = StageEpochRecorder()
    callbacks = [
        recorder,
        tf.keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_accuracy", mode="max",
                                           save_best_only=True),
        tf.keras.callbacks.ModelCheckpoint(str(latest_path)),
        tf.keras.callbacks.CSVLogger(str(run_dir / "epochs.csv"), append=False),
        tf.keras.callbacks.TerminateOnNaN(),
    ]
    if schedule:
        callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                             patience=3, min_lr=1e-7))
    if early_stopping:
        callbacks.append(tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max",
                                                          patience=5, restore_best_weights=False))
    history = model.fit(stage_dataset("train", batch_size, smoke),
                        validation_data=stage_dataset("val", batch_size, smoke),
                        epochs=epochs, callbacks=callbacks, verbose=verbose, shuffle=False).history
    history = {key: [float(value) for value in history[key]] for key in
               ["loss", "accuracy", "val_loss", "val_accuracy", "learning_rate", "seconds"]}
    if not all(np.isfinite(values).all() for values in history.values()):
        raise RuntimeError("Non-finite metrics; recovery checkpoints retained, no completed record saved")
    epochs_run = len(history["loss"])
    if not early_stopping and epochs_run != epochs:
        raise RuntimeError("Stage did not finish its declared budget")
    best_epoch = int(np.argmax(history["val_accuracy"])) + 1
    best_model = tf.keras.models.load_model(best_path, compile=False)
    assert model.count_params() == best_model.count_params()
    record = {
        "status": "completed", "metadata": metadata, "history": history,
        "epochs_run": epochs_run, "best_epoch": best_epoch,
        "best_val_accuracy": history["val_accuracy"][best_epoch - 1],
        "best_val_loss": history["val_loss"][best_epoch - 1],
        "trainable_params": trainable, "total_params": int(model.count_params()),
        "time_per_epoch_s": float(np.mean(recorder.times[1:] or recorder.times)),
        "timing_excludes_first_epoch": len(recorder.times) > 1,
        "best_checkpoint": str(best_path), "latest_checkpoint": str(latest_path),
        "best_checkpoint_sha256": file_sha256(best_path),
        "latest_checkpoint_sha256": file_sha256(latest_path),
    }
    write_stage_json(record_path, record)
    print(f"Saved {stage} stage: {record_path}; best epoch {best_epoch}, trainable params {trainable:,}")
    return record


In [51]:
RUN_STAGE_SMOKE = False  # Explicitly enable a temporary two-stage execution check.
if RUN_STAGE_SMOKE:
    import tempfile

    with tempfile.TemporaryDirectory(prefix="efficientnet_stages_") as temporary_dir:
        root = Path(temporary_dir)
        common = dict(epochs=1, batch_size=2, smoke=True, output_dir=root / "records",
                      checkpoint_dir=root / "checkpoints", verbose=0)
        head_settings = dict(run_id="smoke_head", stage="head", learning_rate=1e-3, **common)
        head_record = run_transfer_stage(**head_settings)
        assert run_transfer_stage(**head_settings) == head_record
        fine_record = run_transfer_stage(
            run_id="smoke_fine", stage="finetune", learning_rate=1e-5, unfreeze_from="block7a",
            head_run=root / "records/smoke_head.json", **common,
        )
        before = tf.keras.models.load_model(head_record["best_checkpoint"], compile=False)
        after = tf.keras.models.load_model(fine_record["latest_checkpoint"], compile=False)
        before_base, after_base = before.get_layer("efficientnetb0"), after.get_layer("efficientnetb0")
        for layer in after_base.layers:
            if isinstance(layer, layers.BatchNormalization):
                original = before_base.get_layer(layer.name)
                np.testing.assert_array_equal(layer.moving_mean.numpy(), original.moving_mean.numpy())
                np.testing.assert_array_equal(layer.moving_variance.numpy(), original.moving_variance.numpy())
                assert not layer.trainable
        np.testing.assert_array_equal(before_base.get_layer("stem_conv").kernel.numpy(),
                                       after_base.get_layer("stem_conv").kernel.numpy())
        assert not np.array_equal(before_base.get_layer("top_conv").kernel.numpy(),
                                  after_base.get_layer("top_conv").kernel.numpy()), "Fine-tuning weights did not update"
        assert fine_record["metadata"]["parent_checkpoint_sha256"] == head_record["best_checkpoint_sha256"]
        try:
            run_transfer_stage(**{**head_settings, "learning_rate": 0.002})
        except ValueError as error:
            assert "Cached stage differs" in str(error)
        else:
            raise AssertionError("Changed settings must reject a completed cache")
    print("Two-stage smoke, cache and frozen-weight checks passed; temporary artifacts removed.")


## 6. Validation experiment design

First train the frozen-backbone head for five epochs with Adam at 0.001. Save its validation-best checkpoint. Then compare four fine-tuning candidates: learning rates 0.0001 and 0.00001, unfreezing from either `block6a` or `block7a`. Every candidate loads the **same best head checkpoint**, uses a fresh optimizer and shuffle state, batch size 4 and a fixed ten-epoch budget. Keep learning rates constant and disable early stopping for these comparisons.

This is a small declared search, not proof of an optimal configuration. Rank candidates by highest validation accuracy, then lower validation loss at that epoch, then run ID. Record the head-only baseline separately: its five epochs are a prerequisite, not an equal-budget fine-tuning candidate. If fine-tuning does not beat the head baseline, report that outcome; do not assume unfreezing helps.

No test inference occurs. Full experiments are disabled by default and reuse only compatible completed records. Recovery checkpoints are ignored by Git, so retain them locally if you want to continue from the trained head. The JSON experiment records and report plots can be committed after actual runs finish.


In [52]:
HEAD_EPOCHS = 5
FINE_EPOCHS = 10
EXPERIMENT_BATCH = 4
HEAD_SETTINGS = dict(run_id="head_adam_lr_001", stage="head", learning_rate=1e-3,
                     epochs=HEAD_EPOCHS, batch_size=EXPERIMENT_BATCH)
FINE_CANDIDATES = [
    dict(run_id="fine_block6a_lr_0001", unfreeze_from="block6a", learning_rate=1e-4),
    dict(run_id="fine_block6a_lr_00001", unfreeze_from="block6a", learning_rate=1e-5),
    dict(run_id="fine_block7a_lr_0001", unfreeze_from="block7a", learning_rate=1e-4),
    dict(run_id="fine_block7a_lr_00001", unfreeze_from="block7a", learning_rate=1e-5),
]
RUN_VALIDATION_EXPERIMENTS = False
RERUN_VALIDATION_EXPERIMENTS = False


In [53]:
head_experiment = None
fine_experiments = {}
if RUN_VALIDATION_EXPERIMENTS:
    head_experiment = run_transfer_stage(**HEAD_SETTINGS, rerun=RERUN_VALIDATION_EXPERIMENTS)
    head_record_path = EXPERIMENT_DIR / (HEAD_SETTINGS["run_id"] + ".json")
    for candidate in FINE_CANDIDATES:
        print("Fine-tuning candidate:", candidate, flush=True)
        fine_experiments[candidate["run_id"]] = run_transfer_stage(
            **candidate, stage="finetune", epochs=FINE_EPOCHS, batch_size=EXPERIMENT_BATCH,
            head_run=head_record_path, schedule=False, early_stopping=False,
            rerun=RERUN_VALIDATION_EXPERIMENTS,
        )
else:
    print("Validation experiments disabled; no full training started.")


Validation experiments disabled; no full training started.


In [54]:
def summarize_validation_candidates(head_record, candidates):
    """Verify comparable completed candidates and return a ranked validation table."""
    if head_record.get("status") != "completed" or head_record["metadata"]["stage"] != "head":
        raise ValueError("Expected a completed head baseline")
    if head_record["metadata"]["smoke"]:
        raise ValueError("Smoke metrics cannot select a full training recipe")
    expected_ids = {candidate["run_id"] for candidate in FINE_CANDIDATES}
    if set(candidates) != expected_ids:
        raise ValueError("All declared fine-tuning candidates must complete before ranking")
    rows = []
    for settings in FINE_CANDIDATES:
        run_id = settings["run_id"]
        record = candidates[run_id]
        metadata = record["metadata"]
        if record.get("status") != "completed" or metadata["stage"] != "finetune" or metadata["smoke"]:
            raise ValueError("Expected completed full fine-tuning runs")
        assert metadata["parent_checkpoint_sha256"] == head_record["best_checkpoint_sha256"], "Different head initialization"
        for key, value in settings.items():
            assert metadata[key] == value, f"Candidate {run_id} differs from declared {key}"
        assert metadata["epochs"] == record["epochs_run"] == FINE_EPOCHS
        assert metadata["batch_size"] == head_record["metadata"]["batch_size"] == EXPERIMENT_BATCH
        assert not metadata["schedule"] and not metadata["early_stopping"]
        for key in ["pretrained", "seed", "class_names", "thread_environment", "devices", "platform"]:
            assert metadata[key] == head_record["metadata"][key], f"Different {key}"
        history = record["history"]
        for key in ["loss", "accuracy", "val_loss", "val_accuracy"]:
            assert len(history[key]) == FINE_EPOCHS and np.isfinite(history[key]).all()
        best = int(np.argmax(history["val_accuracy"]))
        rows.append(dict(
            run_id=run_id, unfreeze_from=metadata["unfreeze_from"], learning_rate=metadata["learning_rate"],
            epochs=record["epochs_run"], trainable_params=record["trainable_params"],
            best_val_accuracy=history["val_accuracy"][best], best_epoch=best + 1,
            val_loss_at_best=history["val_loss"][best], final_val_accuracy=history["val_accuracy"][-1],
            final_val_loss=history["val_loss"][-1],
            last_five_val_accuracy_std=float(np.std(history["val_accuracy"][-5:])),
            time_per_epoch_s=record["time_per_epoch_s"],
            improvement_over_head_pp=100 * (history["val_accuracy"][best] - head_record["best_val_accuracy"]),
        ))
    return pd.DataFrame(rows).sort_values(
        ["best_val_accuracy", "val_loss_at_best", "run_id"], ascending=[False, True, True],
    ).reset_index(drop=True)


def report_validation_candidates(head_record, candidates, output_dir=Path("results"), figures_dir=Path("figures")):
    table = summarize_validation_candidates(head_record, candidates)
    output_dir, figures_dir = Path(output_dir), Path(figures_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir / "sota_efficientnetb0_validation.csv", index=False)
    chosen_id = str(table.loc[0, "run_id"])
    chosen = dict(next(candidate for candidate in FINE_CANDIDATES if candidate["run_id"] == chosen_id))
    write_stage_json(output_dir / "sota_efficientnetb0_validation.json", {
        "head": head_record, "candidates": candidates, "selected_finetuning": chosen,
        "selection_rule": "best val accuracy; then lower loss at that epoch; then run ID",
    })
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for run_id, record in candidates.items():
        history = record["history"]
        epochs = np.arange(1, len(history["loss"]) + 1)
        axes[0].plot(epochs, history["val_loss"], label=run_id)
        axes[1].plot(epochs, history["val_accuracy"], label=run_id)
    axes[0].axhline(head_record["best_val_loss"], color="black", linestyle="--", label="head checkpoint loss")
    axes[1].axhline(head_record["best_val_accuracy"], color="black", linestyle="--", label="head checkpoint accuracy")
    for axis, ylabel in zip(axes, ["Validation cross-entropy", "Validation accuracy"]):
        axis.set_xlabel("Fine-tuning epoch"); axis.set_ylabel(ylabel)
        axis.grid(alpha=0.3); axis.legend(fontsize=8)
    axes[1].set_ylim(0, 1)
    fig.tight_layout(); fig.savefig(figures_dir / "sota_efficientnetb0_validation.png", dpi=150)
    plt.show(); plt.close(fig)
    print(table.to_string(index=False))
    print("Best fine-tuning candidate:", chosen)
    print("Head baseline validation accuracy:", head_record["best_val_accuracy"])
    return table, chosen


if head_experiment is not None and len(fine_experiments) == len(FINE_CANDIDATES):
    validation_table, CHOSEN_FINE = report_validation_candidates(head_experiment, fine_experiments)
else:
    print("No candidate selected: complete the full validation experiments first.")


No candidate selected: complete the full validation experiments first.


### Interpret after actual execution

Compare learning rates at the same unfreezing depth, and unfreezing depths at the same learning rate. Discuss best and final validation accuracy, validation loss, trainable parameters and curve stability. A larger trainable subset increases adaptation capacity and may increase overfitting or runtime; the outcome must come from the observed results.

Do not interpret smoke checks or synthetic reporting tests as accuracy evidence. Do not use the test set to select a learning rate or upper block. Record hardware and batch size when discussing timing, and consider repeated seeds if small validation differences are important.

## Next stage

Execute the declared validation experiments through this notebook, inspect the real curves and record a numerical discussion. Then train the selected final recipe, retain the best checkpoint across both stages and evaluate it once on the untouched test set.
